In [0]:
print("E-Commerce Lakehouse Project Started!")

print("Spark Version:", spark.version)

print("Databricks Environment Ready!")

E-Commerce Lakehouse Project Started!
Spark Version: 4.2.0
Databricks Environment Ready!


In [0]:
# Define the source data location

BASE_PATH = "/Volumes/workspace/default/olist_raw/"

# List all uploaded files

files = dbutils.fs.ls(BASE_PATH)

for file in files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [0]:
from pyspark.sql import functions as F

# Load all nine datasets

customers_df = spark.read.csv(
    BASE_PATH + "olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)

geolocation_df = spark.read.csv(
    BASE_PATH + "olist_geolocation_dataset.csv",
    header=True,
    inferSchema=True
)

order_items_df = spark.read.csv(
    BASE_PATH + "olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

payments_df = spark.read.csv(
    BASE_PATH + "olist_order_payments_dataset.csv",
    header=True,
    inferSchema=True
)

reviews_df = spark.read.csv(
    BASE_PATH + "olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=True
)

orders_df = spark.read.csv(
    BASE_PATH + "olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    BASE_PATH + "olist_products_dataset.csv",
    header=True,
    inferSchema=True
)

sellers_df = spark.read.csv(
    BASE_PATH + "olist_sellers_dataset.csv",
    header=True,
    inferSchema=True
)

category_translation_df = spark.read.csv(
    BASE_PATH + "product_category_name_translation.csv",
    header=True,
    inferSchema=True
)

print("All nine datasets loaded successfully!")

All nine datasets loaded successfully!


In [0]:
datasets = {
    "customers": customers_df,
    "geolocation": geolocation_df,
    "order_items": order_items_df,
    "payments": payments_df,
    "reviews": reviews_df,
    "orders": orders_df,
    "products": products_df,
    "sellers": sellers_df,
    "category_translation": category_translation_df
}

print("Total datasets loaded:", len(datasets))

Total datasets loaded: 9


In [0]:
for name, df in datasets.items():

    print("\n" + "=" * 60)
    print("DATASET:", name.upper())
    print("=" * 60)

    df.printSchema()


DATASET: CUSTOMERS
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)


DATASET: GEOLOCATION
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)


DATASET: ORDER_ITEMS
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)


DATASET: PAYMENTS
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- p

In [0]:
for name, df in datasets.items():

    print(
        name,
        "Rows:", df.count(),
        "Columns:", len(df.columns)
    )

customers Rows: 99441 Columns: 5
geolocation Rows: 1000163 Columns: 5
order_items Rows: 112650 Columns: 7
payments Rows: 103886 Columns: 5
reviews Rows: 104162 Columns: 7
orders Rows: 99441 Columns: 8
products Rows: 32951 Columns: 9
sellers Rows: 3095 Columns: 4
category_translation Rows: 71 Columns: 2


In [0]:
for name, df in datasets.items():

    print("\nDATASET:", name.upper())

    null_counts = df.select([
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)
        for c in df.columns
    ])

    display(null_counts)


DATASET: CUSTOMERS


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,0,0,0,0



DATASET: GEOLOCATION


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,0,0,0,0



DATASET: ORDER_ITEMS


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,0,0,0,0,0,0


... (8 outputs left out)


DATASET: SELLERS


seller_id,seller_zip_code_prefix,seller_city,seller_state
0,0,0,0



DATASET: CATEGORY_TRANSLATION


product_category_name,product_category_name_english
0,0


In [0]:
for name, df in datasets.items():

    total_records = df.count()

    unique_records = df.dropDuplicates().count()

    duplicate_records = total_records - unique_records

    print(f"""
    Dataset: {name}
    Total Records: {total_records}
    Unique Records: {unique_records}
    Duplicate Records: {duplicate_records}
    """)


    Dataset: customers
    Total Records: 99441
    Unique Records: 99441
    Duplicate Records: 0
    

    Dataset: geolocation
    Total Records: 1000163
    Unique Records: 738332
    Duplicate Records: 261831
    

    Dataset: order_items
    Total Records: 112650
    Unique Records: 112650
    Duplicate Records: 0
    

    Dataset: payments
    Total Records: 103886
    Unique Records: 103886
    Duplicate Records: 0
    

    Dataset: reviews
    Total Records: 104162
    Unique Records: 104077
    Duplicate Records: 85
    

    Dataset: orders
    Total Records: 99441
    Unique Records: 99441
    Duplicate Records: 0
    

    Dataset: products
    Total Records: 32951
    Unique Records: 32951
    Duplicate Records: 0
    

    Dataset: sellers
    Total Records: 3095
    Unique Records: 3095
    Duplicate Records: 0
    

    Dataset: category_translation
    Total Records: 71
    Unique Records: 71
    Duplicate Records: 0
    
